In [0]:
%fs ls /Volumes/cinedata/bronze/inputs

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze;

In [0]:
from pyspark.sql.functions import current_timestamp

#caminho para os dados
volume_path = "/Volumes/cinedata/bronze/inputs"

#dicionario de mapeamento das tabelas csv -> db de acordo com o pdf da atividade
tabelas_mapeamento = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

#loop for de leitura e transformacao dos arquivos csv em tabela delta sem alterar o tipo dos dados
for arquivo, nome_tabela in tabelas_mapeamento.items():
    df = spark.read.csv(f"{volume_path}/{arquivo}", header=True, inferSchema=True)
    df = df.withColumn("ingestion_datetime", current_timestamp())
    df.write.format("delta").mode("append").saveAsTable(nome_tabela)
    print(f"Arquivo {arquivo} transformado em tabela {nome_tabela} com sucesso!")

In [0]:
%sql
SELECT * FROM bronze.tb_movies_info LIMIT 10; 